In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import Dataset
from huggingface_hub import login, hf_hub_download
from transformers import (
    PretrainedConfig, 
    PreTrainedTokenizerFast, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import precision_recall_fscore_support
import zipfile
from tqdm.auto import tqdm


In [ ]:
# ==========================================
# 1. AUTHENTICATION & CONFIGURATION
# ==========================================
login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-encdec-checkpoints" # Phase 1 EncDec Checkpoints[cite: 5]
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints" #[cite: 4, 5]
OUTPUT_DIR = "./phase4_encdec_lorma"

def load_cjpe_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_4_CJPE/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 4 CJPE Data...")
train_data = load_cjpe_file("Train/train.json")
dev_data = load_cjpe_file("Train/dev.json")

# Toggle debug mode if a rapid sanity check is needed
DEBUG_MODE = False
if DEBUG_MODE:
    print("⚠️ DEBUG MODE ACTIVE: Subsampling dataset.")
    train_data = train_data[:100]
    dev_data = dev_data[:20]



Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

⚠️ DEBUG MODE ACTIVE: Subsampling dataset.


In [ ]:
# ==========================================
# 2. TOKENIZER & DUAL-ANCHOR CHUNKING
# ==========================================
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"]) #[cite: 4, 5]
tokenizer.padding_side = "right" #[cite: 4, 5]
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token #[cite: 4, 5]

def prepare_dataset_records(raw_records, tokenizer, max_length=512):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for record in raw_records:
        doc_text = record.get("text", "")
        label = record.get("label", 0)
        
        tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
        full_ids = tokenized["input_ids"]
        
        if len(full_ids) <= max_length:
            chunks = [full_ids]
        else:
            first_chunk = full_ids[:max_length]
            last_chunk = full_ids[-max_length:]
            chunks = [first_chunk, last_chunk]
            
        for chunk in chunks:
            input_ids_list.append(chunk)
            attention_mask_list.append([1] * len(chunk))
            labels_list.append(label)
            
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

print("Processing Dataset Features...")
train_features = prepare_dataset_records(train_data, tokenizer)
dev_features = prepare_dataset_records(dev_data, tokenizer)

train_dataset = Dataset.from_dict(train_features)
eval_dataset = Dataset.from_dict(dev_features)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) #[cite: 4, 5]


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Processing Dataset Features...


In [4]:
# ==========================================
# 3. ENCODER-DECODER LORMA ARCHITECTURE
# ==========================================
class CustomLegalConfig(PretrainedConfig):
    model_type = "custom_legal_encdec"
    def __init__(self, vocab_size=32000, d_model=768, num_layers=6, num_heads=12, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff

class LormaAdapter(nn.Module):
    def __init__(self, d_model, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(d_model, rank, bias=False)
        self.B = nn.Linear(rank, d_model, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

class CustomLormaEncoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

    def forward(self, src, src_key_padding_mask=None):
        src2, _ = self.self_attn(src, src, src, key_padding_mask=src_key_padding_mask)
        src2 = src2 + self.lorma_attn(src2)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src2 = src2 + self.lorma_mlp(src2)
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class CustomLormaDecoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_self_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.multihead_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_cross_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.norm3 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, tgt, memory, tgt_mask=None, memory_key_padding_mask=None, tgt_key_padding_mask=None):
        tgt2, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt2 = tgt2 + self.lorma_self_attn(tgt2)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        tgt2, _ = self.multihead_attn(tgt, memory, memory, key_padding_mask=memory_key_padding_mask)
        tgt2 = tgt2 + self.lorma_cross_attn(tgt2)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt2 = tgt2 + self.lorma_mlp(tgt2)
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

class LormaEncDecForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, rank=128):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        self.d_model = config.d_model
        
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(1024, config.d_model)
        
        self.encoder = nn.ModuleList([CustomLormaEncoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        self.decoder = nn.ModuleList([CustomLormaDecoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        
        self.classifier = nn.Linear(config.d_model, num_labels)

    def _get_pos_ids(self, x):
        return torch.arange(x.size(1), device=x.device).unsqueeze(0).expand(x.size(0), -1)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device
        
        # Encoder Pass
        src_pos = self._get_pos_ids(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(src_pos)
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        for layer in self.encoder:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        memory = x
        
        # Decoder Pass 
        tgt_pos = self._get_pos_ids(input_ids)
        tgt = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(tgt_pos)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(input_ids.size(1), device=device)
        
        for layer in self.decoder:
            tgt = layer(
                tgt, 
                memory, 
                tgt_mask=tgt_mask, 
                memory_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=src_key_padding_mask
            )
            
        # Mean Pooling over the decoder output
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(tgt.size()).float()
            sum_embeddings = torch.sum(tgt * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled_logits = sum_embeddings / sum_mask
        else:
            pooled_logits = tgt.mean(dim=1)

        logits = self.classifier(pooled_logits)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)


In [5]:
# ==========================================
# 4. INITIALIZATION & WEIGHT TRANSFER
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing LORMA Encoder-Decoder Binary Classifier for Phase 4...")

encdec_config = CustomLegalConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id
)
model = LormaEncDecForSequenceClassification(encdec_config, num_labels=2, rank=128)

print("Loading and Remapping Phase 1 Weights...")
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN) #[cite: 5]
state_dict = load_file(model_weight_path) #[cite: 4, 5]

mapped_dict = {}
for key, val in state_dict.items():
    if key.startswith("encoder.layers."):
        mapped_dict[key.replace("encoder.layers.", "encoder.")] = val
    elif key.startswith("decoder.layers."):
        mapped_dict[key.replace("decoder.layers.", "decoder.")] = val
    else:
        mapped_dict[key] = val

model.load_state_dict(mapped_dict, strict=False)

# Freeze base model
for param in model.parameters(): 
    param.requires_grad = False

# Unfreeze LORMA adapters + classifier head
for layer in model.encoder:
    for param in layer.lorma_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for layer in model.decoder:
    for param in layer.lorma_self_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_cross_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for param in model.classifier.parameters(): 
    param.requires_grad = True

# FP32 upscaling for stable training under AMP
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Trainable Parameters: {trainable_params:,}")


Initializing LORMA Encoder-Decoder Binary Classifier for Phase 4...
Loading and Remapping Phase 1 Weights...


model.safetensors: reconstructing file:   0%|          |  0.00B /  521MB            

model.safetensors: downloading bytes:           |  0.00B            

Model ready. Trainable Parameters: 5,899,778


In [6]:
# ==========================================
# 5. METRICS & TRAINING EXECUTION
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

training_args_encdec = TrainingArguments(
    output_dir=OUTPUT_DIR,
    report_to="none",
    num_train_epochs=5, 
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,         
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=True,
    seed=42
)

encdec_trainer = Trainer(
    model=model,
    args=training_args_encdec,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\nCommencing Phase 4 LORMA Encoder-Decoder Training...")
encdec_trainer.train()


Commencing Phase 4 LORMA Encoder-Decoder Training...


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch,Training Loss,Validation Loss,Macro Precision,Macro Recall,Macro F1
1,No log,0.756390,0.200000,0.500000,0.285714
2,No log,0.746570,0.200000,0.500000,0.285714
3,No log,0.824640,0.200000,0.500000,0.285714
4,2.537617,0.877155,0.500000,0.500000,0.437340
5,2.537617,1.013815,0.515625,0.510417,0.427083


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


TrainOutput(global_step=65, training_loss=2.4670335036057693, metrics={'train_runtime': 77.1937, 'train_samples_per_second': 12.954, 'train_steps_per_second': 0.842, 'total_flos': 0.0, 'train_loss': 2.4670335036057693, 'epoch': 5.0})

In [7]:
# ==========================================
# 6. TEST INFERENCE & SUBMISSION GENERATION
# ==========================================
print("\nLoading Test Data for Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_4_CJPE/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
model.to(device)
predictions = []

print(f"Running inference on {len(test_data)} test examples...")
for case in tqdm(test_data, desc="Predicting CJPE Cases"):
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    encoded = tokenizer(
        case_text, 
        truncation=False, 
        return_attention_mask=False
    )
    full_ids = encoded["input_ids"]
    chunk = full_ids[:512] # Match max length used during feature chunking
    
    input_ids = torch.tensor([chunk], dtype=torch.long).to(device)
    attention_mask = torch.tensor([[1] * len(chunk)], dtype=torch.long).to(device)
    
    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred_label = int(outputs.logits.argmax(dim=-1).item())
        
    predictions.append({
        "id": case_id,
        "pred": pred_label
    })

os.makedirs(OUTPUT_DIR, exist_ok=True)
pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
sub_zip_path = os.path.join(OUTPUT_DIR, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 4 CJPE Prediction complete! Generated {len(predictions)} predictions.")
print(f"Submission saved to: {sub_zip_path}")


Loading Test Data for Submission Generation...


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Running inference on 1517 test examples...


Predicting CJPE Cases:   0%|          | 0/1517 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(



Phase 4 CJPE Prediction complete! Generated 1517 predictions.
Submission saved to: ./phase4_encdec_lorma/submission.zip
